# Scene Classification with a CNN

Computer Vision course project by **Isabel Gregorio Díez** and **Carlota Campos Rubio**.

The goal is to classify natural scene images into six categories: buildings, forest, glacier, mountain, sea and street. The notebook covers data preparation, augmentation, CNN training, evaluation and a simple confidence-based rejection rule for uncertain predictions.


## 0. Download the dataset


In [ ]:
import kagglehub
from pathlib import Path

dataset_path = Path(
    kagglehub.dataset_download("nitishabharathi/scene-classification")
)

print("Dataset downloaded to:", dataset_path)


## 1. Load libraries and define configuration


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Hyperparameters
IMG_SIZE = 150
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3

# Reject predictions below this confidence.
CONFIDENCE_THRESHOLD = 0.80
OPEN_SET_LABEL = "None of the above"

# Dataset paths
DATA_ROOT = dataset_path
IMAGE_DIR = DATA_ROOT / "train-scene classification" / "train"
CSV_PATH = DATA_ROOT / "train-scene classification" / "train.csv"
TEST_CSV_PATH = DATA_ROOT / "test_WyRytb0.csv"

print("Train images folder exists:", IMAGE_DIR.exists())
print("Train CSV exists:", CSV_PATH.exists())
print("Test CSV exists:", TEST_CSV_PATH.exists())

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
BEST_MODEL_PATH = OUTPUT_DIR / "best_scene_cnn.pth"
CHECKPOINT_PATH = OUTPUT_DIR / "scene_classification_checkpoint.pth"

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


## 2. Load the CSV and inspect the dataset


In [ ]:
# Check that the expected dataset files exist.
print("DATA_ROOT =", DATA_ROOT)
print("IMAGE_DIR =", IMAGE_DIR)
print("CSV_PATH  =", CSV_PATH)

assert os.path.isdir(IMAGE_DIR), f"IMAGE_DIR does not exist: {IMAGE_DIR}"
assert os.path.isfile(CSV_PATH), f"CSV_PATH does not exist: {CSV_PATH}"

# Read the training CSV.
df = pd.read_csv(CSV_PATH)

print("CSV shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())


## 3. Detect columns and create image paths


In [ ]:
# Find a column by trying several possible names.
def find_column(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None

image_col = find_column(df.columns, ["image_name", "filename", "image", "img", "file", "id"])
label_col = find_column(df.columns, ["label", "class", "category", "scene"])

print("Detected image column:", image_col)
print("Detected label column:", label_col)

assert image_col is not None, f"Image column not found. Available columns: {list(df.columns)}"
assert label_col is not None, f"Label column not found. Available columns: {list(df.columns)}"

# Add a default .jpg extension when the CSV stores image ids without extensions.
def ensure_image_extension(value):
    value = str(value)
    if Path(value).suffix.lower() in IMG_EXTENSIONS:
        return value
    return value + ".jpg"

df["image_file"] = df[image_col].apply(ensure_image_extension)
df["image_path"] = df["image_file"].apply(lambda x: os.path.join(IMAGE_DIR, x))

# Keep only rows whose image file actually exists.
df = df[df["image_path"].apply(os.path.exists)].copy()

print("Rows after filtering existing images:", len(df))
display(df[[image_col, label_col, "image_file", "image_path"]].head())


## 4. Create readable class names and split the data


In [ ]:
# Standard label names for the Intel Scene Classification dataset.
# The original CSV usually stores labels as numeric ids from 0 to 5.
INTEL_SCENE_LABELS = {
    0: "buildings",
    1: "forest",
    2: "glacier",
    3: "mountain",
    4: "sea",
    5: "street",
}

# Convert labels to readable names when they match the known Intel label ids.
label_numeric = pd.to_numeric(df[label_col], errors="coerce")
unique_numeric_labels = sorted(label_numeric.dropna().astype(int).unique().tolist())

if len(unique_numeric_labels) > 0 and set(unique_numeric_labels).issubset(set(INTEL_SCENE_LABELS.keys())):
    df["original_label_id"] = label_numeric.astype(int)
    df["label_name"] = df["original_label_id"].map(INTEL_SCENE_LABELS)
    classes = [INTEL_SCENE_LABELS[i] for i in unique_numeric_labels]
    original_to_idx = {original_id: idx for idx, original_id in enumerate(unique_numeric_labels)}
    df["label_idx"] = df["original_label_id"].map(original_to_idx)
else:
    df["label_name"] = df[label_col].astype(str)
    classes = sorted(df["label_name"].unique().tolist())
    class_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}
    df["label_idx"] = df["label_name"].map(class_to_idx)

idx_to_class = {idx: class_name for idx, class_name in enumerate(classes)}
class_to_idx = {class_name: idx for idx, class_name in idx_to_class.items()}

print("Classes:", classes)
print("Number of classes:", len(classes))
print(df["label_name"].value_counts())

# Stratified split: 70% train, 15% validation, 15% test.
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label_idx"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label_idx"],
    random_state=SEED
)

train_samples = list(zip(train_df["image_path"].tolist(), train_df["label_idx"].tolist()))
val_samples = list(zip(val_df["image_path"].tolist(), val_df["label_idx"].tolist()))
test_samples = list(zip(test_df["image_path"].tolist(), test_df["label_idx"].tolist()))

print(f"Train samples: {len(train_samples)}")
print(f"Validation samples: {len(val_samples)}")
print(f"Test samples: {len(test_samples)}")


## 5. Define transforms, dataset, and dataloaders


In [ ]:
# Training transforms include light data augmentation.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# Evaluation transforms must be deterministic.
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

class SceneDataset(Dataset):
    # Custom PyTorch dataset for image paths and numeric labels.
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, int(label)

trainset = SceneDataset(train_samples, transform=train_transform)
valset = SceneDataset(val_samples, transform=eval_transform)
testset = SceneDataset(test_samples, transform=eval_transform)

train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(valset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoaders are ready.")


## 6. Show random training samples


In [ ]:
# Undo normalization to display images correctly.
def unnormalize_image(tensor):
    image = tensor / 2 + 0.5
    image = image.clamp(0, 1)
    return image

def show_random_samples(loader, classes, num_samples=5):
    images, labels = next(iter(loader))
    num_samples = min(num_samples, len(images))

    fig, axes = plt.subplots(1, num_samples, figsize=(15, 4))
    if num_samples == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        img = unnormalize_image(images[i]).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(classes[labels[i].item()])
        ax.axis("off")

    plt.show()

show_random_samples(train_loader, classes, num_samples=5)


## 7. Define the custom CNN


In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.adapt = nn.AdaptiveAvgPool2d((4, 4))

        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.dropout1 = nn.Dropout(0.5)

        self.fc2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(0.5)

        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))

        x = self.adapt(x)
        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.dropout1(x)

        x = F.relu(self.fc2(x))
        x = self.dropout2(x)

        x = self.fc3(x)
        return x

model = CNN(num_classes=len(classes)).to(device)
print(model)


## 8. Define loss, optimizer, and scheduler


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


## 9. Train the model with validation and early stopping


In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS, patience=5):
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(epochs):
        # Training phase.
        model.train()
        running_train_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        avg_train_loss = running_train_loss / len(train_loader)
        train_acc = 100 * correct_train / total_train

        # Validation phase.
        model.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item()
                predicted = torch.argmax(outputs, dim=1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        avg_val_loss = running_val_loss / len(val_loader)
        val_acc = 100 * correct_val / total_val

        scheduler.step(avg_val_loss)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
            f"LR: {optimizer.param_groups[0]['lr']:.6f}"
        )

        # Save the model with the best validation loss.
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print("Best model saved.")
        else:
            epochs_without_improvement += 1

        # Stop early if validation loss does not improve.
        if epochs_without_improvement >= patience:
            print(f"Early stopping activated at epoch {epoch + 1}.")
            break

    return train_losses, val_losses, train_accuracies, val_accuracies

train_losses, val_losses, train_accuracies, val_accuracies = train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    patience=5
)

print("Training finished.")


## 10. Evaluate on the normal test set


In [ ]:
def load_best_model(model, model_path=BEST_MODEL_PATH):
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

def evaluate_closed_set(model, test_loader):
    model = load_best_model(model)

    correct = 0
    total = 0
    test_loss = 0.0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            predicted = torch.argmax(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_test_loss = test_loss / len(test_loader)
    test_accuracy = 100 * correct / total

    print(f"Closed-set Test Loss: {avg_test_loss:.4f}")
    print(f"Closed-set Test Accuracy: {test_accuracy:.2f}%")

    return avg_test_loss, test_accuracy

closed_test_loss, closed_test_accuracy = evaluate_closed_set(model, test_loader)


## 11. Evaluate the confidence threshold on known test images


In [ ]:
def evaluate_with_rejection(model, test_loader, threshold=CONFIDENCE_THRESHOLD):
    # This test set contains only known classes.
    # Therefore, if a known image is rejected as "None of the above",
    # it is counted as an error for final accuracy.

    model = load_best_model(model)

    total = 0
    accepted = 0
    rejected = 0

    correct_closed_set = 0
    correct_when_accepted = 0
    correct_final = 0

    y_true = []
    y_pred_closed = []
    y_pred_threshold = []
    confidences = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            probabilities = F.softmax(logits, dim=1)

            confidence, predicted = torch.max(probabilities, dim=1)

            for i in range(labels.size(0)):
                true_idx = labels[i].item()
                pred_idx = predicted[i].item()
                conf = confidence[i].item()

                true_class = classes[true_idx]
                closed_pred_class = classes[pred_idx]

                if conf >= threshold:
                    final_pred_class = closed_pred_class
                else:
                    final_pred_class = OPEN_SET_LABEL

                y_true.append(true_class)
                y_pred_closed.append(closed_pred_class)
                y_pred_threshold.append(final_pred_class)
                confidences.append(conf)

                total += 1

                if pred_idx == true_idx:
                    correct_closed_set += 1

                if conf >= threshold:
                    accepted += 1
                    if pred_idx == true_idx:
                        correct_when_accepted += 1
                        correct_final += 1
                else:
                    rejected += 1
                    # Rejected known-class images count as errors in this test set.

    closed_set_accuracy = 100 * correct_closed_set / total
    final_accuracy = 100 * correct_final / total
    coverage = 100 * accepted / total
    rejection_rate = 100 * rejected / total
    accepted_accuracy = 100 * correct_when_accepted / accepted if accepted > 0 else 0.0

    print(f"Threshold: {threshold:.2f}")
    print(f"Closed-set accuracy without threshold: {closed_set_accuracy:.2f}%")
    print(f"Final accuracy with threshold: {final_accuracy:.2f}%")
    print(f"Accepted known images: {accepted}/{total} ({coverage:.2f}%)")
    print(f"Rejected known images: {rejected}/{total} ({rejection_rate:.2f}%)")
    print(f"Accuracy on accepted images only: {accepted_accuracy:.2f}%")

    return {
        "threshold": threshold,
        "closed_set_accuracy": closed_set_accuracy,
        "final_accuracy": final_accuracy,
        "coverage": coverage,
        "rejection_rate": rejection_rate,
        "accepted_accuracy": accepted_accuracy,
        "y_true": y_true,
        "y_pred_closed": y_pred_closed,
        "y_pred_threshold": y_pred_threshold,
        "confidences": confidences,
    }


threshold_report = evaluate_with_rejection(
    model,
    test_loader,
    threshold=CONFIDENCE_THRESHOLD
)


In [ ]:
def plot_confusion_matrices(threshold_report, classes):
    # Confusion matrix without threshold.
    cm_closed = confusion_matrix(
        threshold_report["y_true"],
        threshold_report["y_pred_closed"],
        labels=classes
    )

    disp_closed = ConfusionMatrixDisplay(
        confusion_matrix=cm_closed,
        display_labels=classes
    )

    plt.figure(figsize=(8, 8))
    disp_closed.plot(cmap="Blues", xticks_rotation=45)
    plt.title("Confusion Matrix - Closed-set Classification")
    plt.show()

    # Confusion matrix with threshold.
    # The predictions may include "None of the above".
    threshold_labels = classes + [OPEN_SET_LABEL]

    cm_threshold = confusion_matrix(
        threshold_report["y_true"],
        threshold_report["y_pred_threshold"],
        labels=threshold_labels
    )

    disp_threshold = ConfusionMatrixDisplay(
        confusion_matrix=cm_threshold,
        display_labels=threshold_labels
    )

    plt.figure(figsize=(9, 8))
    disp_threshold.plot(cmap="Blues", xticks_rotation=45)
    plt.title(f"Confusion Matrix - With Confidence Threshold ({CONFIDENCE_THRESHOLD})")
    plt.show()


plot_confusion_matrices(threshold_report, classes)


## 12. Plot training curves


In [ ]:
# Plot training and validation loss.
plt.figure(figsize=(7, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()

# Plot training and validation accuracy.
plt.figure(figsize=(7, 5))
plt.plot(train_accuracies, label="Training Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()
plt.show()


## 13. Show predictions on random test images with confidence


In [ ]:
def predict_tensor_with_threshold(model, image_tensor, classes, threshold=CONFIDENCE_THRESHOLD):
    model.eval()

    with torch.no_grad():
        logits = model(image_tensor.to(device))
        probabilities = F.softmax(logits, dim=1)
        confidence, predicted_idx = torch.max(probabilities, dim=1)

    confidence = confidence.item()
    predicted_idx = predicted_idx.item()
    closest_known_class = classes[predicted_idx]

    if confidence < threshold:
        final_prediction = OPEN_SET_LABEL
    else:
        final_prediction = closest_known_class

    return final_prediction, closest_known_class, confidence

def show_predictions_with_threshold(model, test_loader, classes, threshold=CONFIDENCE_THRESHOLD, num_samples=5):
    model = load_best_model(model)

    images, labels = next(iter(test_loader))
    num_samples = min(num_samples, len(images))

    fig, axes = plt.subplots(1, num_samples, figsize=(18, 4))
    if num_samples == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        img = unnormalize_image(images[i]).permute(1, 2, 0).numpy()

        final_prediction, closest_known_class, confidence = predict_tensor_with_threshold(
            model,
            images[i:i+1],
            classes,
            threshold=threshold
        )

        true_class = classes[labels[i].item()]
        is_correct = final_prediction == true_class

        ax.imshow(img)
        ax.set_title(
            f"True: {true_class}\n"
            f"Pred: {final_prediction}\n"
            f"Closest: {closest_known_class}\n"
            f"Conf: {confidence:.2f}",
            color="green" if is_correct else "red"
        )
        ax.axis("off")

    plt.show()

show_predictions_with_threshold(
    model,
    test_loader,
    classes,
    threshold=CONFIDENCE_THRESHOLD,
    num_samples=5
)


## 14. Test an external image

Use `predict_image_or_unknown()` with any image path. If the maximum class probability is below the confidence threshold, the output is `"None of the above"`.


In [ ]:
def predict_image_or_unknown(
    image_path,
    model,
    classes,
    transform=eval_transform,
    threshold=CONFIDENCE_THRESHOLD,
    display_image=True
):
    model = load_best_model(model)

    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        probabilities = F.softmax(logits, dim=1)
        confidence, predicted_idx = torch.max(probabilities, dim=1)

    confidence = confidence.item()
    predicted_idx = predicted_idx.item()

    closest_known_class = classes[predicted_idx]
    final_prediction = closest_known_class if confidence >= threshold else OPEN_SET_LABEL

    print(f"Image path: {image_path}")
    print(f"Final prediction: {final_prediction}")
    print(f"Closest known class: {closest_known_class}")
    print(f"Confidence: {confidence:.4f}")
    print(f"Threshold: {threshold:.2f}")

    print("\nClass probabilities:")
    probs_np = probabilities.squeeze(0).cpu().numpy()
    for class_name, prob in sorted(zip(classes, probs_np), key=lambda x: x[1], reverse=True):
        print(f"  {class_name}: {prob:.4f}")

    if display_image:
        plt.figure(figsize=(6, 6))
        plt.imshow(image)
        plt.axis("off")
        plt.title(
            f"Prediction: {final_prediction}\n"
            f"Closest known class: {closest_known_class} | Confidence: {confidence:.2f}"
        )
        plt.show()

    return {
        "final_prediction": final_prediction,
        "closest_known_class": closest_known_class,
        "confidence": confidence,
        "threshold": threshold,
        "probabilities": dict(zip(classes, probs_np.tolist())),
    }


In [ ]:
# Example:
# external_result = predict_image_or_unknown(
#     image_path="path/to/image.jpg",
#     model=model,
#     classes=classes,
#     threshold=CONFIDENCE_THRESHOLD,
# )


## 15. Visualize filters from the first convolutional layer


In [ ]:
def visualize_filters(model):
    model = load_best_model(model)
    filters = model.conv1.weight.data.cpu().numpy()

    fig, axes = plt.subplots(4, 8, figsize=(10, 5))

    for i, ax in enumerate(axes.flat):
        if i < filters.shape[0]:
            img = filters[i].transpose(1, 2, 0)
            img_min = img.min()
            img_max = img.max()
            img = (img - img_min) / (img_max - img_min + 1e-8)
            ax.imshow(img)
        ax.axis("off")

    plt.suptitle("First Convolutional Layer Filters")
    plt.show()

visualize_filters(model)


## 16. Save the final checkpoint


In [ ]:
model = load_best_model(model)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "classes": classes,
    "class_to_idx": class_to_idx,
    "idx_to_class": idx_to_class,
    "img_size": IMG_SIZE,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "open_set_label": OPEN_SET_LABEL,
}

torch.save(checkpoint, CHECKPOINT_PATH)

print(f"Checkpoint saved to: {CHECKPOINT_PATH}")
